# FinSight — Phase 1 Walkthrough: Ingestion & Baseline RAG

This notebook replays **every step of Phase 1 by hand** so you can see the raw
data, what each transformation does, and why each design decision was made.

**Pipeline:**

```
EDGAR API → raw HTML → clean text → named sections → metadata chunks
                                                          ↓
   cited answer ← GPT ← vector search ← pgvector ← embeddings
```

## Prerequisites

1. `pip install -r requirements.txt` into `.venv` (already done if you followed the README)
2. `.env` filled with your Azure OpenAI endpoint/key
3. Postgres running: `docker compose -f infra/docker-compose.yml up -d`

> Steps 1–4 (download → parse → chunk) need **no API key and cost nothing** —
> you can run them offline as many times as you like. Steps 5–8 call Azure
> OpenAI (fractions of a cent per run, except the one-time full embedding load).


In [1]:
# Setup: make repo modules importable from the notebooks/ folder
import sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_colwidth", 100)

from config import AZURE_OPENAI_ENDPOINT, EDGAR_USER_AGENT, DATABASE_URL
print("repo root      :", ROOT)
print("edgar UA       :", EDGAR_USER_AGENT)
print("azure endpoint :", AZURE_OPENAI_ENDPOINT or "(not set — steps 5-8 will fail)")

repo root      : c:\Users\subha\OneDrive\Desktop\Subham\finsight
edgar UA       : FinSight research project subham.tiwari186@gmail.com
azure endpoint : https://finsight-st.openai.azure.com/


---
## Step 1 — Raw data: the SEC EDGAR API

Everything starts with two free JSON endpoints (no key, but the SEC **requires**
a `User-Agent` header identifying you, and caps you at ~10 requests/second):

| Endpoint | Gives you |
|---|---|
| `sec.gov/files/company_tickers.json` | ticker → CIK (SEC's company ID) |
| `data.sec.gov/submissions/CIK{cik}.json` | every filing the company made |

Let's look at both **raw**, before any of our code touches them.

In [2]:
import httpx

client = httpx.Client(headers={"User-Agent": EDGAR_USER_AGENT}, timeout=30, follow_redirects=True)

# --- raw ticker map: it's a dict of row-number -> record ---
tickers_raw = client.get("https://www.sec.gov/files/company_tickers.json").json()
print("total companies listed:", len(tickers_raw))
print(json.dumps(dict(list(tickers_raw.items())[:3]), indent=2))

total companies listed: 10426
{
  "0": {
    "cik_str": 1045810,
    "ticker": "NVDA",
    "title": "NVIDIA CORP"
  },
  "1": {
    "cik_str": 320193,
    "ticker": "AAPL",
    "title": "Apple Inc."
  },
  "2": {
    "cik_str": 1652044,
    "ticker": "GOOGL",
    "title": "Alphabet Inc."
  }
}


In [3]:
# Find JPMorgan's CIK (Central Index Key)
jpm = next(v for v in tickers_raw.values() if v["ticker"] == "JPM")
cik = jpm["cik_str"]
print(jpm)

# --- raw submissions index for JPM (note: only the ~1000 most recent filings) ---
subs = client.get(f"https://data.sec.gov/submissions/CIK{cik:010d}.json").json()
recent = subs["filings"]["recent"]
print("\ncompany:", subs["name"])
print("filings in 'recent' window:", len(recent["form"]))
print("distinct form types:", len(set(recent["form"])))

{'cik_str': 19617, 'ticker': 'JPM', 'title': 'JPMORGAN CHASE & CO'}

company: JPMORGAN CHASE & CO
filings in 'recent' window: 25444
distinct form types: 26


In [4]:
# The raw index is parallel arrays — one entry per filing. Put it in a
# DataFrame and filter to the form types FinSight cares about.
filings_df = pd.DataFrame({
    "form": recent["form"],
    "filingDate": recent["filingDate"],
    "reportDate": recent["reportDate"],
    "primaryDocument": recent["primaryDocument"],
    "accessionNumber": recent["accessionNumber"],
})
filings_df[filings_df.form.isin(["10-K", "10-Q", "8-K"])].head(10)

,form,filingDate,reportDate,primaryDocument,accessionNumber
332,8-K,2026-07-14,2026-07-14,jpm-20260714.htm,0001628280-26-048086
341,8-K,2026-07-14,2026-07-14,jpm-20260714.htm,0001628280-26-048078
1659,8-K,2026-06-25,2026-06-24,jpm-20260624.htm,0000019617-26-000241
1681,8-K,2026-06-24,2026-06-24,jpm-20260624.htm,0001628280-26-045169
1682,8-K,2026-06-24,2026-06-24,jpm-20260624.htm,0001628280-26-045167
3163,8-K,2026-06-02,2026-06-02,d113536d8k.htm,0001193125-26-253533
3827,8-K,2026-05-27,2026-05-27,d51433d8k.htm,0001193125-26-242018
4074,8-K,2026-05-21,2026-05-19,jpm-20260519.htm,0000019617-26-000228
5079,8-K,2026-05-07,2026-05-06,d903351d8k.htm,0001193125-26-211978
5654,10-Q,2026-05-01,2026-03-31,jpm-20260331.htm,0001628280-26-029344


In [5]:
# Download the latest 10-K with our client (ingestion/edgar_client.py adds
# rate-limiting + retries + local caching on top of what we did by hand above).
from ingestion.edgar_client import EdgarClient

edgar = EdgarClient()
filing = edgar.list_filings("JPM", forms=("10-K",), since="2024-01-01")[0]
raw_path = edgar.download_filing(filing)   # cached — instant if already downloaded

raw_html = raw_path.read_bytes()
print(f"{filing.form} for period {filing.report_date}")
print(f"file: {raw_path.name}  ({len(raw_html)/1e6:.1f} MB of HTML)")

10-K for period 2025-12-31
file: 10-K_2025-12-31.htm  (12.9 MB of HTML)


In [6]:
# THIS is what a modern 10-K actually looks like on disk: inline-XBRL soup.
# The first ~600 chars:
print(raw_html[:600].decode("utf-8", "replace"))

<?xml version='1.0' encoding='ASCII'?>
<!--XBRL Document Created with the Workiva Platform-->
<!--Copyright 2026 Workiva-->
<!--r:c7652ff4-626c-4654-b145-19dfe5912602,g:d35e7381-7855-4836-bd1e-2a303b8cc81d,d:1e82dc5e49024170976eb7ddc7c0a10b-->
<html xmlns="http://www.w3.org/1999/xhtml" xmlns:ixt="http://www.xbrl.org/inlineXBRL/transformation/2020-02-12" xmlns:ixt-sec="http://www.sec.gov/inlineXBRL/transformation/2015-08-31" xmlns:ix="http://www.xbrl.org/2013/inlineXBRL" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:dei="http://xbrl.sec.gov/dei/2025" xmlns:us-gaap="http://fasb.org


**Takeaway:** the raw filing is 12+ MB of machine-generated XBRL/HTML — styling
attributes everywhere, content buried in nested `<div>`s and `<span>`s. Nobody
can chunk this directly; first we need clean text.

---
## Step 2 — Preprocessing: HTML → clean text

`ingestion/parser.py::html_to_text` does three things:

1. drops `<script>`/`<style>`
2. **flattens tables into pipe-separated rows** — so a number keeps its row
   label as context ("Provision for credit losses | 9,320 | 6,899")
3. collapses whitespace but **keeps line breaks** (heading detection needs them)

Watch it work on a toy example first, then on the real filing.

In [7]:
from ingestion.parser import html_to_text

toy = '''<html><body>
  <style>.x{color:red}</style>
  <h1>Item 1A. Risk Factors</h1>
  <p>The Firm faces <span style="font-weight:bold">many</span> risks.</p>
  <table>
    <tr><th>Metric</th><th>2025</th><th>2024</th></tr>
    <tr><td>Provision for credit losses</td><td>9,320</td><td>6,899</td></tr>
  </table>
</body></html>'''
print(html_to_text(toy))

Item 1A. Risk Factors

The Firm faces 
many
 risks.

Metric | 2025 | 2024
Provision for credit losses | 9,320 | 6,899


In [8]:
%%time
# Now the real 12.9 MB filing (takes ~10-20s — lxml has to parse the whole tree)
text = html_to_text(raw_html)
print(f"{len(raw_html):,} bytes of HTML  ->  {len(text):,} chars of text")

12,927,325 bytes of HTML  ->  1,449,219 chars of text
CPU times: total: 2.06 s
Wall time: 2.07 s


In [9]:
# Same content, before vs after — find the Risk Factors heading in clean text
i = text.find("Item 1A. Risk Factors.")
print(text[i:i+700])

Item 1A. Risk Factors. 
The following discussion sets forth the material risk factors that could affect JPMorganChase’s financial condition and operations. Readers should not consider any descriptions of these factors to be a complete set of all potential risks that could affect the Firm. Any of the risk factors discussed below could by itself, or combined with other factors, materially and adversely affect JPMorganChase’s business, results of operations, financial condition, capital position, liquidity, competitive position or reputation, including by materially increasing expenses or decreasing revenues, which could result in material losses or a decrease in earnings.
Summary
The principal


---
## Step 3 — Splitting into named sections

10-Ks have a standard structure (`Item 1A` = Risk Factors, `Item 7` = MD&A, ...).
We find headings with a regex, **but there's a trap**: every item also appears
in the table of contents. The fix in `parser.split_items`:

- a match followed by `< 200` chars before the next match is a TOC entry → drop it
- if an item still matches twice, keep the **longest** occurrence

In [10]:
from ingestion.parser import _ITEM_RE, split_items

# The regex, and every raw match — note the early cluster: that's the TOC
matches = list(_ITEM_RE.finditer(text))
print("pattern:", _ITEM_RE.pattern)
print(f"{len(matches)} raw 'Item N' matches; first 12 positions:")
for m in matches[:12]:
    print(f"   pos {m.start():>9,}  Item {m.group(1)}")

pattern: (?:^|\n)\s*ITEM\s+(\d{1,2}[A-C]?)\s*[.:—\-]
43 raw 'Item N' matches; first 12 positions:
   pos   238,695  Item 1
   pos   238,950  Item 1A
   pos   238,982  Item 1B
   pos   239,059  Item 2
   pos   239,086  Item 3
   pos   239,120  Item 4
   pos   239,168  Item 5
   pos   239,293  Item 6
   pos   239,317  Item 7
   pos   239,419  Item 7A
   pos   239,495  Item 8
   pos   239,555  Item 9


In [11]:
sections = split_items(text, "10-K")
pd.DataFrame([
    {"item": s.item, "name": s.name[:60], "chars": len(s.text)} for s in sections
])

,item,name,chars
0,1,Business,39406
1,1A,Risk Factors,112633
2,2,Properties,2148
3,5,Market for Registrant's Common Equity,2747
4,7,Management's Discussion and Analysis (MD&A),395
5,7A,Quantitative and Qualitative Disclosures About Market Risk,270
6,8,Financial Statements and Supplementary Data,370
7,9A,Controls and Procedures,2229
8,9B,Other Information,5493
9,10,"Directors, Executive Officers and Corporate Governance",3996


**Note the two quirks** (both documented in the README):

- **Item 7 (MD&A) is tiny** — JPM "incorporates it by reference", meaning the
  actual MD&A text physically lives inside the giant **Item 15** exhibit blob.
  The content is still indexed, just under the Item 15 label.
- **Item 15 is ~1M chars** — it holds the MD&A + full financial statements.

---
## Step 4 — Section-aware chunking with metadata headers

Embedding models have input limits and retrieval works best on focused passages,
so each section is split into ~4,000-char chunks on **paragraph boundaries**,
with a **one-paragraph overlap** so a sentence at a boundary is never orphaned.

The key design decision: every chunk gets a **metadata header** —
`[JPM | 10-K FY2025 | Item 1A: Risk Factors]` — so both the embedding vector
and the LLM see *what this text is*, and a ready-made **citation** travels with
the chunk from ingestion all the way to the final answer.

In [12]:
from ingestion.chunker import chunk_section

risk = next(s for s in sections if s.item == "1A")
chunks = chunk_section(risk, "JPM", "10-K", filing.report_date)

sizes = [len(c.text) for c in chunks]
print(f"Item 1A: {len(risk.text):,} chars -> {len(chunks)} chunks")
print(f"chunk sizes: min {min(sizes):,} / mean {sum(sizes)//len(sizes):,} / max {max(sizes):,} chars")
print("citation on every chunk, e.g.:", chunks[0].citation)

Item 1A: 112,633 chars -> 24 chunks
chunk sizes: min 3,260 / mean 9,427 / max 10,743 chars
citation on every chunk, e.g.: [JPM 10-K 2025, Item 1A]


In [13]:
# A full chunk, exactly as it will be embedded (header + body):
print(chunks[3].text[:1200], "\n[...truncated...]")

[JPM | 10-K FY2025 | Item 1A: Risk Factors]
that are stricter than a global standard, which could create competitive disadvantages for those firms, such as JPMorganChase, that are subject to the enhanced regulations. Furthermore, certain authorities outside the U.S. have adopted applicable law that could conflict with or prohibit JPMorganChase from complying with applicable law in other jurisdictions, which could create conflict of law issues and could increase risks associated with non-compliance.
Regulatory initiatives outside the U.S. have required and could in the future require JPMorganChase to significantly modify its operations or legal entity structure in the places in which those initiatives are implemented, such as requirements for: 
•
establishing locally-based intermediate holding companies or operating subsidiaries
•
maintaining minimum amounts of capital or liquidity in locally-based subsidiaries
•
implementing processes within locally-based subsidiaries for complying wit

In [14]:
# The overlap: last paragraph of chunk N == first paragraph of chunk N+1
tail = chunks[3].text.split("\n\n")[-1]
head = chunks[4].text.split("\n")[1:]  # skip metadata header line
print("chunk 3 ends with :", tail[:120], "...")
print("chunk 4 starts with:", "\n".join(head)[:120], "...")

chunk 3 ends with : Part I
collateral consequences for the subject financial institution, including:
•
loss of clients, customers and busine ...
chunk 4 starts with: Part I
collateral consequences for the subject financial institution, including:
•
loss of clients, customers and busine ...


---
## Step 5 — Embeddings *(Azure OpenAI key required from here on)*

`text-embedding-3-large` turns text into a **3072-dimensional vector** where
similar meanings land near each other. That's what makes "search by meaning"
possible — the model knows *loan loss provision* and *credit losses* are related
even though they share no keywords.

In [15]:
from ingestion.embedder import get_openai, embed_texts

aoai = get_openai()
demo = [
    "The provision for credit losses increased due to loan deterioration.",
    "Reserves set aside for loans that may not be repaid grew this quarter.",  # same meaning, different words
    "Our new headquarters features a rooftop garden for employees.",           # unrelated
]
vecs = embed_texts(aoai, demo)
print(f"{len(vecs)} vectors, {len(vecs[0])} dims each. First 6 dims of vec 0:")
print([round(x, 4) for x in vecs[0][:6]])

3 vectors, 3072 dims each. First 6 dims of vec 0:
[-0.0453, 0.0071, -0.0025, 0.0245, 0.0061, -0.0208]


In [16]:
# Cosine similarity: the two credit-risk sentences should score far higher
# with each other than with the rooftop garden.
import math

def cosine(a, b):
    dot = sum(x*y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x*x for x in a)) * math.sqrt(sum(y*y for y in b)))

for i in range(3):
    for j in range(i+1, 3):
        print(f"sim(text{i}, text{j}) = {cosine(vecs[i], vecs[j]):.3f}   "
              f"| {demo[i][:40]}... vs {demo[j][:40]}...")

sim(text0, text1) = 0.604   | The provision for credit losses increase... vs Reserves set aside for loans that may no...
sim(text0, text2) = 0.071   | The provision for credit losses increase... vs Our new headquarters features a rooftop ...
sim(text1, text2) = 0.132   | Reserves set aside for loans that may no... vs Our new headquarters features a rooftop ...


---
## Step 6 — Storing in Postgres + pgvector

Schema (`infra/schema.sql`): one row per chunk with its metadata, the 3072-dim
`vector` column, and a generated `tsvector` column that Phase 2 will use for
BM25 keyword search. The `UNIQUE (ticker, form, period, item, seq)` constraint
makes the loader **idempotent** — re-running it never duplicates rows.

The bulk load is `python -m ingestion.embedder` (takes ~10+ min on a trial-tier
rate limit for ~700 chunks). The cell below only loads if the table is empty.

In [17]:
import subprocess
import psycopg

with psycopg.connect(DATABASE_URL) as conn:
    n = conn.execute("SELECT count(*) FROM chunks").fetchone()[0]

if n == 0:
    print("table empty -> running full embed+load (this takes a while)...")
    subprocess.run([sys.executable, "-m", "ingestion.embedder"], cwd=ROOT, check=True)
else:
    print(f"chunks table already loaded: {n} rows — skipping embed step")

with psycopg.connect(DATABASE_URL) as conn:
    rows = conn.execute(
        "SELECT ticker, form, fiscal_label, count(*) FROM chunks GROUP BY 1,2,3 ORDER BY 1"
    ).fetchall()
pd.DataFrame(rows, columns=["ticker", "form", "period", "chunks"])

chunks table already loaded: 731 rows — skipping embed step


,ticker,form,period,chunks
0,BAC,10-K,FY2025,256
1,JPM,10-K,FY2025,475


---
## Step 7 — Retrieval: vector search with metadata filters

Query flow: embed the question with the *same* model → cosine-distance search
(`<=>` operator) → optional SQL filters on ticker/section. Filters matter: for
"JPMorgan's risks" you don't want Bank of America chunks winning on similarity.

In [18]:
from retrieval.search import search

hits = search("How exposed is the bank to cyber attacks and ransomware?", k=5, tickers=["JPM"])
pd.DataFrame([
    {"score": round(h.score, 3), "citation": h.citation,
     "section": h.section_name[:40], "preview": h.text[len(h.citation)+30:160]}
    for h in hits
])

,score,citation,section,preview
0,0.561,"[JPM 10-K 2025, Item 1A]",Risk Factors,"ade by JPMorganChase or another market participant, whether inadvertent or malicious, which coul..."
1,0.538,"[JPM 10-K 2025, Item 15]",Exhibits and Financial Statement Schedul,"tatement Schedules (incl. content incorporated by reference: MD&A, financial statements)]\nManag..."
2,0.524,"[JPM 10-K 2025, Item 1A]",Risk Factors,"ctions by banking regulators, as well as changes in applicable law or how applicable law is impl..."
3,0.507,"[JPM 10-K 2025, Item 1A]",Risk Factors,"organChase or its clients, customers, counterparties or employees\n•\nmanipulate or destroy data..."
4,0.493,"[JPM 10-K 2025, Item 15]",Exhibits and Financial Statement Schedul,"tatement Schedules (incl. content incorporated by reference: MD&A, financial statements)]\nThe C..."


Scores ~0.6–0.75 mean strong matches (for an out-of-scope question they drop
to ~0.25 — you'll see that in Step 8).

---
## Step 8 — RAG: grounded answers with enforced citations

The system prompt is the control layer — it's what separates this from a
chatbot. Priority-ordered rules: **only** use provided evidence, cite every
claim, and say *"Insufficient evidence"* rather than guess.

In [19]:
from rag_cli import SYSTEM_PROMPT
print(SYSTEM_PROMPT)

You are FinSight, a financial research assistant answering questions about
SEC filings. Rules, in priority order:

1. Use ONLY the evidence passages provided below. Never use outside
   knowledge or memory for facts or figures.
2. Every factual claim MUST end with the bracketed citation of the passage
   it came from, e.g. [JPM 10-K 2025, Item 1A]. No citation -> do not make
   the claim.
3. If the evidence does not answer the question, reply exactly:
   "Insufficient evidence in the indexed filings." and say what is missing.
4. Write in the concise, neutral tone of an equity research analyst.



In [20]:
from IPython.display import Markdown
from config import CHAT_DEPLOYMENT

def rag_answer(question, tickers=None, k=8):
    hits = search(question, k=k, tickers=tickers)
    evidence = "\n\n---\n\n".join(
        f"PASSAGE {i+1} {h.citation}:\n{h.text}" for i, h in enumerate(hits)
    )
    resp = aoai.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"EVIDENCE:\n\n{evidence}\n\nQUESTION: {question}"},
        ],
    )
    return resp.choices[0].message.content, hits

answer, hits = rag_answer("What risks did JPMorgan flag about artificial intelligence?", tickers=["JPM"])
Markdown(answer)

- AI system failures, inappropriate use, lack of transparency, or inaccurate/biased outputs could disrupt operations, cause erroneous transactions, compromise data privacy, infringe intellectual property, harm clients and impair JPMorganChase’s decision‑making [JPM 10-K 2025, Item 1A].  
- Poorly designed or insufficiently safeguarded AI—especially agentic systems—could increase exposure to cyber attacks, system manipulation or data loss by allowing systems to access sensitive data or take actions without proper controls [JPM 10-K 2025, Item 1A].  
- AI can intensify cyber threats by enabling malicious actors to develop more advanced social‑engineering attacks, reverse‑engineer security patches or otherwise facilitate unauthorized access and data breaches [JPM 10-K 2025, Item 1A].  
- Rapidly evolving legal and regulatory regimes for AI, including inconsistent international standards, could raise compliance costs, lead to fines or sanctions, and restrict JPMorganChase’s use of AI technologies [JPM 10-K 2025, Item 1A].  
- Competitors that deploy AI more quickly or effectively could gain cost, client‑experience or product‑innovation advantages, causing JPMorganChase to lose market share; AI agents could also disintermediate direct customer relationships [JPM 10-K 2025, Item 1A].  
- Miscalibrated workforce planning related to AI (including over‑reliance or failure to adopt) could produce staff shortages, hinder development of critical employee skills, or prevent JPMorganChase from capturing AI efficiencies [JPM 10-K 2025, Item 1A].  
- Rapid deployment without adequate testing, poor data or ineffective model design and controls can produce flawed models and biased outputs, increasing operational, compliance and model‑risk exposures [JPM 10-K 2025, Item 1A].  
- AI‑related incidents or AI‑generated misinformation could damage reputation, trigger litigation or regulatory action, and lead to heightened scrutiny or loss of clients and investors [JPM 10-K 2025, Item 1A].

In [21]:
# The trust test: a question the index CANNOT answer. A plain chatbot would
# happily hallucinate Tesla facts; FinSight must refuse.
answer, hits = rag_answer("What did Tesla say about vehicle recall risks?", tickers=["JPM", "BAC"])
print("top retrieval score:", round(hits[0].score, 3), "(vs ~0.7 for on-topic questions)")
Markdown(answer)

top retrieval score: 0.275 (vs ~0.7 for on-topic questions)


Insufficient evidence in the indexed filings.

The provided materials are excerpts from JPMorgan Chase & Co.’s 2025 Form 10‑K, not Tesla filings [JPM 10‑K 2025, Item 15]. Missing: Tesla’s SEC filings or public statements addressing vehicle recall risks.

---
## Recap — what you built, step by step

| Step | Input → Output | Key decision |
|---|---|---|
| 1 | EDGAR API → 12.9 MB raw HTML | rate limit + User-Agent per SEC policy |
| 2 | HTML → clean text | flatten tables to keep row context |
| 3 | text → named sections | drop TOC duplicates, keep longest match |
| 4 | sections → ~4K-char chunks | **metadata header + citation on every chunk** |
| 5 | chunks → 3072-dim vectors | search by meaning, not keywords |
| 6 | vectors → pgvector rows | idempotent load; `tsv` column ready for Phase 2 |
| 7 | question → top-k chunks | metadata filters + cosine similarity |
| 8 | chunks → cited answer | system prompt enforces cite-or-refuse |

**Phase 2 next:** golden dataset (~50 Q&A) → RAGAS eval harness → hybrid
search (BM25 + reciprocal rank fusion on the `tsv` column) → cross-encoder
reranker — measuring every change against the baseline you just built.